# cli

> Simple CLI for LLM agents — fetch, search, and read commands.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
"""CLI for fossick — fetch, search, and read commands for LLM agents.
Default output is readable markdown; pass --as-json for JSON (piping/harnesses)."""

import json, sys
from fastcore.script import call_parse, anno_parser
from fossick.core import (fetch as _fetch, to_md, read_arxiv as _read_arxiv,
                          read_yt as _read_yt, search_yt as _search_yt,
                          url2nb as _url2nb, pdf2nb as _pdf2nb, download_yt as _download_yt,
                          crawl as _crawl, find_xhr as _find_xhr, paginate_api as _paginate_api,
                          read_gh_repo as _read_gh_repo, read_gh_file as _read_gh_file,
                          lookup_doi as _lookup_doi, mv_skill_md, repo_root)
from fossick.search import (search as _search, images as _images, news as _news,
                            videos as _videos, google as _google, research as _research)


In [ ]:
#| export
@call_parse
def fetch(
    url:str,              # URL to fetch
    sel:str=None,         # CSS selector to extract (None = full page)
    heavy:bool=False,     # JS rendering via headless browser
    stealthy:bool=False,  # Anti-bot stealth fetcher
    session:bool=False,   # Route through the persistent debug Chrome (reuses its logged-in cookies)
    auto:bool=False,      # Auto-escalate plain->heavy->stealthy->session on bot-block detection
    as_json:bool=False,   # Output JSON instead of markdown
):
    "Fetch a URL and print as markdown."
    page = _fetch(url, sel=sel, heavy=heavy, stealthy=stealthy, session=session, auto=auto)
    if as_json: print(json.dumps({'url': page['url'], 'status': page['status'], 'tier': getattr(page,'tier',None), 'content': to_md(page)}))
    else: print(to_md(page))

In [ ]:
#| export
@call_parse
def crawl(
    url:str,                    # start URL
    sel:str=None,              # CSS selector to extract per page
    follow_sel:str='a[href]',  # CSS selector for links to follow
    max_pages:int=10,          # max pages to visit
    same_domain:bool=True,     # only follow same-domain links
    heavy:bool=False,          # JS rendering via headless browser
    stealthy:bool=False,       # anti-bot stealth fetcher
    as_json:bool=False,        # output JSON
):
    "Crawl from a URL following links; prints each page as markdown."
    pages = _crawl(url, sel=sel, follow_sel=follow_sel, max_pages=max_pages,
                   same_domain=same_domain, heavy=heavy, stealthy=stealthy)
    if as_json: print(json.dumps([{'url': p['url'], 'status': p['status'], 'content': to_md(p, sel=sel)} for p in pages], default=str))
    else:
        for p in pages:
            print(f"# {p['url']}")
            print(to_md(p, sel=sel))
            print()

In [ ]:
#| export
@call_parse
def search(
    query:str,            # search query
    n:int=10,             # number of results
    region:str='us-en',   # ddgs region (us-en, uk-en, ru-ru, ...)
    google:bool=False,    # use direct Google via stealth browser instead of ddgs metasearch
    as_json:bool=False,   # output JSON
):
    "Search the web (ddgs metasearch, or direct Google with --google); prints title, URL, and snippet."
    results = _google(query, n=n) if google else _search(query, max_results=n, region=region)
    if as_json: print(json.dumps(list(results), default=str)); return
    for r in results:
        print(f"## {r.get('title','')}")
        print(f"URL: {r.get('href','')}")
        if (body := r.get('body') or r.get('content')): print(body[:200])
        print()

In [ ]:
#| export
@call_parse
def read_arxiv(
    url:str,              # arXiv URL or paper ID
    source:bool=False,    # include full paper text
    chars:int=4000,       # max source chars to include
    force:bool=False,     # re-download even if cached
    as_json:bool=False,   # output JSON
):
    "Fetch arXiv paper metadata, authors, and summary."
    p = _read_arxiv(url, force=force)
    if as_json:
        out = {k: v for k, v in p.items() if k != 'source'}
        if source: out['source'] = (p.get('source') or '')[:chars]
        print(json.dumps(out, default=str))
    else:
        print(f"# {p['title']}")
        print(f"Authors: {', '.join(p.get('authors', []))}")
        print(f"Published: {str(p.get('published', ''))[:10]}")
        print(f"\n{p.get('summary', '')}")
        if source and p.get('source'): print(f"\n---\n{p['source'][:chars]}")

In [ ]:
#| export
@call_parse
def lookup_doi(
    title:str,            # paper title to look up
    as_json:bool=False,   # output JSON
):
    "Return the doi.org URL for the first Crossref match on a paper title."
    doi = _lookup_doi(title)
    if as_json: print(json.dumps({'title': title, 'doi': doi}))
    else: print(doi or 'not found')

In [ ]:
#| export
@call_parse
def read_yt(
    url:str,              # YouTube URL or video ID
    force:bool=False,     # re-fetch even if cached
    as_json:bool=False,   # output JSON
):
    "Fetch YouTube metadata and full transcript."
    v = _read_yt(url, force=force)
    if as_json: print(json.dumps(v, default=str))
    else:
        print(f"# {v['title']}")
        print(f"Channel: {v['channel']}  Duration: {v.get('duration')}s")
        if v.get('source'): print(f"\n{v['source']}")

In [ ]:
#| export
@call_parse
def search_yt(
    query:str,            # search query
    n:int=10,             # number of results
    as_json:bool=False,   # output JSON
):
    "Search YouTube; prints title, URL, and channel."
    results = _search_yt(query, n=n)
    if as_json: print(json.dumps(list(results), default=str))
    else:
        for r in results:
            print(f"## {r['title']}")
            print(f"URL: {r['url']}  Channel: {r.get('channel', '')}  Duration: {r.get('duration')}s")
            if r.get('description'): print(r['description'][:150])
            print()

In [ ]:
#| export
@call_parse
def read_gh_file(
    url:str,              # GitHub blob URL of the file
    as_json:bool=False,   # output JSON
):
    "Read the raw contents of a single GitHub file."
    txt = _read_gh_file(url)
    if as_json: print(json.dumps({'url': url, 'content': txt}))
    else: print(txt)


In [ ]:
#| export
@call_parse
def read_gh_repo(
    url:str,              # GitHub repo URL, SSH address, or local path
    globs:str=None,       # comma-sep glob patterns (default: README*,pyproject.toml,*.py)
    limit:int=None,       # max files to return
    as_json:bool=False,   # output JSON
):
    "Read files from a GitHub repo filtered by glob patterns."
    gs = tuple(g.strip() for g in globs.split(',')) if globs else None
    files = _read_gh_repo(url, globs=gs, limit=limit)
    if as_json: print(json.dumps({str(k): v for k, v in files.items()}, default=str))
    else:
        for path, content in files.items():
            print(f"# {path}")
            print(content)
            print()

In [ ]:
#| export
@call_parse
def calls(
    url:str,              # URL to navigate to
    pattern:str='.*',     # regex/glob to filter request URLs
    tail:int=3,           # seconds to wait after navigation
    as_json:bool=False,   # output JSON
):
    "Capture outgoing network requests fired by a page."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect()
        return await cdp.calls(url=url, pattern=pattern, tail=tail)
    caps = syncy(_run())
    if as_json: print(json.dumps(list(caps.values()), default=str))
    else:
        for r in caps.values():
            print(f"## {r['url']}")
            if r.get('response_body'): print(str(r['response_body'])[:300])
            print()


In [ ]:
#| export
@call_parse
def find_xhr(
    url:str,              # URL to visit with a headless browser
    pattern:str='*',      # glob/regex to filter captured XHR URLs
    as_json:bool=False,   # output JSON
):
    "Discover hidden JSON/XHR API calls a page makes."
    xhr = _find_xhr(url, pattern=pattern)
    if as_json: print(json.dumps([dict(x) for x in xhr], default=str))
    else:
        for x in xhr:
            print(f"## {x['url']}")
            print(f"type: {x.get('content_type')}")
            if x.get('data'): print(str(x['data'])[:300])
            print()

In [ ]:
#| export
@call_parse
def paginate_api(
    url:str,                      # API endpoint URL
    payload:str=None,             # request body/params as a JSON string
    page_field:str='pageNumber',  # payload key to increment per page
    results_field:str=None,       # response key with the items list (auto-detect if None)
    method:str='POST',            # HTTP method
    max_pages:int=10,             # max pages to fetch
):
    "Paginate a JSON API and print all collected items as JSON."
    body = json.loads(payload) if payload else None
    items = _paginate_api(url, payload=body, page_field=page_field,
                          results_field=results_field, method=method, max_pages=max_pages)
    print(json.dumps(items, default=str))

In [ ]:
#| export
@call_parse
def collect(
    url:str,              # URL to open in Chrome
    save_dir:str='.',     # directory to save screenshots
    tout:int=None,        # stop after this many seconds
    count:int=None,       # stop after this many screenshots
    every_n:int=None,     # auto-capture every N seconds
):
    "Open a URL in Chrome and capture screenshots interactively."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect()
        page = await cdp.open_page(url)
        return await page.collect(save_dir=save_dir, tout=tout, count=count, every_n=every_n)
    shots = syncy(_run())
    print(f'{len(shots)} screenshot(s) saved to {save_dir}')


In [ ]:
#| export
@call_parse
def annotate(
    url:str,              # URL to open in Chrome
    save_dir:str='.',     # directory to save annotated screenshot
    as_json:bool=False,   # output JSON list of selected elements
):
    "Open a URL in Chrome, click elements to annotate them, save labeled screenshot."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect()
        page = await cdp.open_page(url)
        return await page.annotate(save_dir=save_dir)
    img, elements = syncy(_run())
    if as_json: print(json.dumps(elements, default=str))
    else:
        for e in elements: print(f"{e['n']}. {e['role']} · {e['selector']}" + (f" — {e['name']}" if e['name'] else ''))


In [ ]:
#| export
@call_parse
def install():
    "Install fossick SKILL.md to .agents/skills/fossick/ and .claude/skills/fossick/."
    mv_skill_md(dry_run=False, dir=repo_root())

In [ ]:
#| export
@call_parse
def url2nb(
    url:str,              # URL to convert (HTML page, PDF, or arXiv)
    path:str=None,        # output notebook path (default: derived from URL)
    as_json:bool=False,   # output JSON with notebook path
):
    "Convert a URL (HTML, PDF, or arXiv) to a Jupyter notebook."
    nb_path = _url2nb(url, nb_path=path)
    if as_json: print(json.dumps({'path': str(nb_path)}))
    else: print(nb_path)


In [ ]:
#| export
@call_parse
def pdf2nb(
    url:str,              # PDF URL or local path
    path:str=None,        # output notebook path (default: derived from PDF)
    ocr:str='auto',       # OCR mode: auto | on | off
    force:bool=False,     # re-convert even if notebook exists
    as_json:bool=False,   # output JSON with notebook path
):
    "Convert a PDF (URL or local path) to a Jupyter notebook."
    nb_path = _pdf2nb(url, nb_path=path, ocr_selection=ocr, force=force)
    if as_json: print(json.dumps({'path': str(nb_path)}))
    else: print(nb_path)


In [ ]:
#| export
@call_parse
def download_yt(
    url:str,              # YouTube URL or video ID
    format:str='audio',   # audio | video | yt-dlp format string
    save_dir:str='.',     # directory to save to
    as_json:bool=False,   # output JSON with saved path
):
    "Download YouTube audio or video."
    path = _download_yt(url, format=format, save_dir=save_dir)
    if as_json: print(json.dumps({'path': str(path)}))
    else: print(path)


In [ ]:
#| export
@call_parse
def images(query:str, n:int=20, region:str='us-en', as_json:bool=False):
    "Image search via ddgs; prints title and image URL."
    results = _images(query, max_results=n, region=region)
    if as_json: print(json.dumps(list(results), default=str))
    else:
        for r in results: print(f"{r.get('title','')}\n  {r.get('image','')}")

@call_parse
def news(query:str, n:int=20, region:str='us-en', as_json:bool=False):
    "News search via ddgs; prints date, title, and URL."
    results = _news(query, max_results=n, region=region)
    if as_json: print(json.dumps(list(results), default=str))
    else:
        for r in results:
            print(f"## {r.get('title','')}")
            print(f"{r.get('date','')}  {r.get('url','')}")
            if r.get('body'): print(r['body'][:200])
            print()

@call_parse
def videos(query:str, n:int=20, region:str='us-en', as_json:bool=False):
    "Video search via ddgs; prints title and video URL."
    results = _videos(query, max_results=n, region=region)
    if as_json: print(json.dumps(list(results), default=str))
    else:
        for r in results: print(f"{r.get('title','')}\n  {r.get('content','')}")

In [ ]:
#| export
@call_parse
def research(
    query:str,            # search query
    n:int=5,              # number of top results to read
    google:bool=False,    # use direct Google ranking (stealth browser) instead of ddgs metasearch
    sel:str=None,         # CSS selector to narrow each page before markdown
    chars:int=4000,       # max markdown chars per source
    as_json:bool=False,   # output JSON
):
    "Search, then read the top results into one cited markdown corpus."
    res = _research(query, n=n, engine='google' if google else 'search', sel=sel, chars=chars)
    if as_json: print(json.dumps(res, default=str)); return
    print(f"# Research: {res['query']}\n")
    print(res['digest'])

@call_parse
def ax(
    url:str,              # URL to open in the debug Chrome
    port:int=9223,        # debug Chrome remote-debugging port
    full:bool=False,      # show the full accessibility tree instead of interactive-only
):
    "Open a URL in the persistent debug Chrome and print a compact, agent-ready accessibility snapshot."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect(port=port)
        pg = await cdp.open_page(url)
        return await pg.snapshot(interactive=not full)
    print(syncy(_run()))

In [ ]:
#| export
CMDS = {
    'fetch':        fetch,
    'search':       search,
    'research':     research,
    'images':       images,
    'news':         news,
    'videos':       videos,
    'crawl':        crawl,
    'read-arxiv':   read_arxiv,
    'read-yt':      read_yt,
    'search-yt':    search_yt,
    'read-gh-file': read_gh_file,
    'read-gh-repo': read_gh_repo,
    'lookup-doi':   lookup_doi,
    'url2nb':       url2nb,
    'pdf2nb':       pdf2nb,
    'download-yt':  download_yt,
    'calls':        calls,
    'find-xhr':     find_xhr,
    'paginate-api': paginate_api,
    'collect':      collect,
    'annotate':     annotate,
    'ax':           ax,
    'install':      install,
}

def main():
    "Entry point for the `fossick` CLI command."
    if len(sys.argv) < 2 or sys.argv[1] not in CMDS:
        print(f"Usage: fossick [{'  | '.join(CMDS)}]")
        sys.exit(0 if len(sys.argv) < 2 else 1)
    cmd = sys.argv.pop(1)
    func = CMDS[cmd]
    raw  = getattr(func, '__wrapped__', func)      # unwrap @call_parse
    args = anno_parser(raw).parse_args().__dict__  # parse remaining argv
    args.pop('xtra', None); args.pop('pdb', None)  # fastcore-injected extras
    raw(**args)


## Tests

`research`/`ax` should be registered in `CMDS` and `research` importable (it raised `NameError` before the import fix). The `eval:false` cell then runs the `research` command end-to-end against live services.

In [ ]:
# regression guard: research/ax registered and research importable
# (fossick research raised NameError before _research was added to the search import)
import inspect
assert CMDS['research'] is research and CMDS['ax'] is ax
assert _research is not None
assert {'session','auto'} <= set(inspect.signature(fetch.__wrapped__).parameters)
assert {'url','port','full'} <= set(inspect.signature(ax.__wrapped__).parameters)

In [ ]:
#| eval: false
# end-to-end: the research CLI runs against live services and prints a cited corpus
import io
from contextlib import redirect_stdout
buf = io.StringIO()
with redirect_stdout(buf): research.__wrapped__('what is the raft consensus algorithm', n=2)
out = buf.getvalue()
assert out.startswith('# Research:') and 'http' in out
print(out[:300])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()